In [5]:
# prepare data
import pickle 
import numpy as np
with open("./release_assets/metadata/new_hsp60/centers3.pkl", "rb") as f:
    hsp60_centers = pickle.load(f)

In [10]:
hsp60_centers
for i in hsp60_centers:
    array = np.array(hsp60_centers[i]) 
    array = array[:, [2, 1, 0]]
    np.savetxt(f"/virtual_data/test_data/dxd20240920/DeepETPicker/hsp60/raw_data/{i}.txt", array, delimiter="\t")

In [17]:
from pathlib import Path
import json
import subprocess
import sys

repo = Path("./release_assets/external/DeepETPicker")
base = Path("/virtual_data/test_data/dxd20240920/DeepETPicker/hsp60")
raw = base / "raw_data"
coords_dir = raw / "coords"
cfg_dir = base / "configs"
cfg_dir.mkdir(parents=True, exist_ok=True)
runner_python = Path("python")

train_ids = "0,1,2,3,4"
val_id = "5"

# Keep the notebook split explicit so the training and test steps are reproducible.
tomos = sorted(p.stem for p in raw.glob("*.mrc"))
coords = sorted(p.stem for p in coords_dir.glob("*.coords"))
print("Tomograms:", tomos)
print("Coords:", coords)
assert tomos == coords, "Tomogram and coord file stems do not match"

pre_config = {
    "dset_name": "hsp60_preprocess",
    "base_path": str(base),
    "coord_path": str(coords_dir),
    "coord_format": ".coords",
    "tomo_path": str(raw),
    "tomo_format": ".mrc",
    "num_cls": 1,
    "label_type": "gaussian",
    "label_diameter": 11,
    "ocp_type": "sphere",
    "ocp_diameter": "23",
    "norm_type": "standardization",
}

pre_cfg_path = base / "hsp60_preprocess.py"
pre_cfg_path.write_text("pre_config=" + json.dumps(pre_config, indent=12))
print(f"Saved preprocessing config to {pre_cfg_path}")
print(f"Train ids: {train_ids}")
print(f"Val/Test id: {val_id}")

def run_cmd(cmd, log_name):
    log_path = base / log_name
    with log_path.open("w") as log_file:
        subprocess.run(
            cmd,
            cwd=repo / "bin",
            stdout=log_file,
            stderr=subprocess.STDOUT,
            check=True,
        )
    tail = log_path.read_text().splitlines()[-20:]
    print(f"Finished {log_name}")
    if tail:
        print("\n".join(tail))
    return log_path

Tomograms: ['24092001', '24092002', '24092018', '24092024', '24092028', '24092031']
Coords: ['24092001', '24092002', '24092018', '24092024', '24092028', '24092031']
Saved preprocessing config to /virtual_data/test_data/dxd20240920/DeepETPicker/hsp60/hsp60_preprocess.py
Train ids: 0,1,2,3,4
Val/Test id: 5


In [11]:
run_cmd([runner_python, str(repo / "bin" / "preprocess.py"), "--pre_configs", str(pre_cfg_path)], "preprocess.log")

Finished preprocess.log
0 to 2: 477
0 to 3: 633
0 to 4: 810
0 to 5: 866
0 to 6: 1008
Coord generation finished!
****************************************************************************************************
['24092001.mrc', '24092002.mrc', '24092018.mrc', '24092024.mrc', '24092028.mrc', '24092031.mrc']
4/6 finished.
3/6 finished.
2/6 finished.
5/6 finished.
6/6 finished.
1/6 finished.
Standardization finished!
****************************************************************************************************
Label generation finished!
****************************************************************************************************
Occupancy generation finished!
****************************************************************************************************


PosixPath('/virtual_data/test_data/dxd20240920/DeepETPicker/hsp60/preprocess.log')

In [18]:
run_cmd([
    runner_python,
    str(repo / "bin" / "generate_train_config.py"),
    "--pre_configs",
    str(pre_cfg_path),
    "--dset_name",
    "hsp60_train",
    "--cfg_save_path",
    str(cfg_dir),
    "--train_set_ids",
    train_ids,
    "--val_set_ids",
    val_id,
    "--batch_size",
    "8",
    "--block_size",
    "72",
    "--pad_size",
    "12",
    "--learning_rate",
    "0.001",
    "--max_epoch",
    "60",
    "--threshold",
    "0.5",
    "--gpu_id",
    "0",
], "generate_train_config.log")
train_cfg_path = cfg_dir / "hsp60_train.py"
print(f"Training config: {train_cfg_path}")
print(train_cfg_path.read_text().splitlines()[0])

Finished generate_train_config.log
0
Training config: /virtual_data/test_data/dxd20240920/DeepETPicker/hsp60/configs/hsp60_train.py
train_configs={"dset_name": "hsp60_train",


In [24]:
run_cmd([runner_python, str(repo / "bin" / "train_bash.py"), "--train_configs", str(train_cfg_path)], "train.log")

Finished train.log

Validation DataLoader 0:  78%|███████▊  | 14/18 [00:01<00:00, 11.96it/s]

Validation DataLoader 0:  83%|████████▎ | 15/18 [00:01<00:00, 12.07it/s]

Validation DataLoader 0:  89%|████████▉ | 16/18 [00:01<00:00, 12.18it/s]

Validation DataLoader 0:  94%|█████████▍| 17/18 [00:01<00:00, 12.26it/s]

Validation DataLoader 0: 100%|██████████| 18/18 [00:01<00:00, 12.47it/s]

                                                                        
Epoch 59: 100%|██████████| 109/109 [00:32<00:00,  3.31it/s, v_num=3]`Trainer.fit` stopped: `max_epochs=60` reached.

Epoch 59: 100%|██████████| 109/109 [00:32<00:00,  3.31it/s, v_num=3]
****************************************************************************************************
Training Finished
Training pid:2692568
****************************************************************************************************


PosixPath('/virtual_data/test_data/dxd20240920/DeepETPicker/hsp60/train.log')

In [29]:
ckpt_candidates = sorted((base / "runs" / "hsp60_train").glob("**/*.ckpt"), key=lambda p: p.stat().st_mtime)
print("Checkpoint candidates:")
for candidate in ckpt_candidates:
    print(candidate)
assert ckpt_candidates, "No checkpoint was produced by training"
best_ckpt = ckpt_candidates[-1]
print(f"Using checkpoint: {best_ckpt}")

run_cmd([
    runner_python,
    str(repo / "bin" / "test_bash.py"),
    "--train_configs",
    str(train_cfg_path),
    "--checkpoints",
    str(best_ckpt),
], "test.log")

Checkpoint candidates:
/virtual_data/test_data/dxd20240920/DeepETPicker/hsp60/runs/hsp60_train/hsp60_train_ResUNet_BlockSize72_DiceLoss_MaxEpoch60_bs8_lr0.001_IP1_bg1_coord1_Softmax0_bn__TNNone/version_2/checkpoints/epoch=15-step=1744.ckpt
/virtual_data/test_data/dxd20240920/DeepETPicker/hsp60/runs/hsp60_train/hsp60_train_ResUNet_BlockSize72_DiceLoss_MaxEpoch60_bs8_lr0.001_IP1_bg1_coord1_Softmax0_bn__TNNone/version_3/checkpoints/epoch=7-step=872.ckpt
Using checkpoint: /virtual_data/test_data/dxd20240920/DeepETPicker/hsp60/runs/hsp60_train/hsp60_train_ResUNet_BlockSize72_DiceLoss_MaxEpoch60_bs8_lr0.001_IP1_bg1_coord1_Softmax0_bn__TNNone/version_3/checkpoints/epoch=7-step=872.ckpt
Finished test.log
Testing DataLoader 0: 100%|██████████| 666/666 [00:46<00:00, 14.45it/s]
****************************************************************************************************
Testing Finished!
****************************************************************************************************


PosixPath('/virtual_data/test_data/dxd20240920/DeepETPicker/hsp60/test.log')

In [30]:
import csv
import time
from datetime import datetime


def run_cmd_capture(cmd, log_name):
    log_path = base / log_name
    start = time.time()
    with log_path.open("w") as log_file:
        proc = subprocess.run(
            cmd,
            cwd=repo / "bin",
            stdout=log_file,
            stderr=subprocess.STDOUT,
            check=False,
        )
    duration_sec = round(time.time() - start, 2)
    lines = log_path.read_text().splitlines()
    tail = lines[-20:]
    print(f"[{log_name}] exit={proc.returncode}, duration={duration_sec}s")
    if tail:
        print("\n".join(tail))
    return proc.returncode, log_path, duration_sec


all_ids = list(range(len(tomos)))
existing_holdout = int(val_id)
holdouts_to_run = [idx for idx in all_ids if idx != existing_holdout]

results = []

for holdout in holdouts_to_run:
    train_ids_loop = ",".join(str(i) for i in all_ids if i != holdout)
    val_id_loop = str(holdout)
    dset_name_loop = f"hsp60_train_holdout_{holdout}"
    train_cfg_loop = cfg_dir / f"{dset_name_loop}.py"

    print("=" * 100)
    print(f"Running split: holdout={holdout}, test_tomo={tomos[holdout]}, train_ids={train_ids_loop}")

    gen_cmd = [
        runner_python,
        str(repo / "bin" / "generate_train_config.py"),
        "--pre_configs",
        str(pre_cfg_path),
        "--dset_name",
        dset_name_loop,
        "--cfg_save_path",
        str(cfg_dir),
        "--train_set_ids",
        train_ids_loop,
        "--val_set_ids",
        val_id_loop,
        "--batch_size",
        "8",
        "--block_size",
        "72",
        "--pad_size",
        "12",
        "--learning_rate",
        "0.001",
        "--max_epoch",
        "60",
        "--threshold",
        "0.5",
        "--gpu_id",
        "0",
    ]
    gen_code, gen_log, gen_sec = run_cmd_capture(gen_cmd, f"generate_train_config_holdout_{holdout}.log")

    split_result = {
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "holdout_id": holdout,
        "holdout_tomogram": tomos[holdout],
        "train_ids": train_ids_loop,
        "val_id": val_id_loop,
        "dset_name": dset_name_loop,
        "train_config": str(train_cfg_loop),
        "generate_log": str(gen_log),
        "generate_exit": gen_code,
        "generate_seconds": gen_sec,
        "train_log": "",
        "train_exit": None,
        "train_seconds": None,
        "checkpoint": "",
        "test_log": "",
        "test_exit": None,
        "test_seconds": None,
        "status": "pending",
    }

    if gen_code != 0:
        split_result["status"] = "generate_failed"
        results.append(split_result)
        continue

    train_cmd = [
        runner_python,
        str(repo / "bin" / "train_bash.py"),
        "--train_configs",
        str(train_cfg_loop),
    ]
    train_code, train_log, train_sec = run_cmd_capture(train_cmd, f"train_holdout_{holdout}.log")
    split_result["train_log"] = str(train_log)
    split_result["train_exit"] = train_code
    split_result["train_seconds"] = train_sec

    if train_code != 0:
        split_result["status"] = "train_failed"
        results.append(split_result)
        continue

    ckpt_candidates = sorted((base / "runs" / dset_name_loop).glob("**/*.ckpt"), key=lambda p: p.stat().st_mtime)
    if not ckpt_candidates:
        split_result["status"] = "no_checkpoint"
        results.append(split_result)
        continue

    best_ckpt_loop = ckpt_candidates[-1]
    split_result["checkpoint"] = str(best_ckpt_loop)
    print(f"Using checkpoint: {best_ckpt_loop}")

    test_cmd = [
        runner_python,
        str(repo / "bin" / "test_bash.py"),
        "--train_configs",
        str(train_cfg_loop),
        "--checkpoints",
        str(best_ckpt_loop),
    ]
    test_code, test_log, test_sec = run_cmd_capture(test_cmd, f"test_holdout_{holdout}.log")
    split_result["test_log"] = str(test_log)
    split_result["test_exit"] = test_code
    split_result["test_seconds"] = test_sec
    split_result["status"] = "ok" if test_code == 0 else "test_failed"

    results.append(split_result)

results_json = base / "leave_one_out_results_5_splits.json"
results_json.write_text(json.dumps(results, indent=2))

results_csv = base / "leave_one_out_results_5_splits.csv"
with results_csv.open("w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(results[0].keys()) if results else [
        "timestamp", "holdout_id", "holdout_tomogram", "train_ids", "val_id", "dset_name",
        "train_config", "generate_log", "generate_exit", "generate_seconds", "train_log", "train_exit",
        "train_seconds", "checkpoint", "test_log", "test_exit", "test_seconds", "status"
    ])
    writer.writeheader()
    writer.writerows(results)

print("\nSaved summary files:")
print(results_json)
print(results_csv)
print("\nResults:")
for r in results:
    print(r)

Running split: holdout=0, test_tomo=24092001, train_ids=1,2,3,4,5
[generate_train_config_holdout_0.log] exit=0, duration=0.14s
0
[train_holdout_0.log] exit=0, duration=1777.78s

Validation DataLoader 0:  90%|████████▉ | 35/39 [00:02<00:00, 13.13it/s]

Validation DataLoader 0:  92%|█████████▏| 36/39 [00:02<00:00, 13.16it/s]

Validation DataLoader 0:  95%|█████████▍| 37/39 [00:02<00:00, 13.18it/s]

Validation DataLoader 0:  97%|█████████▋| 38/39 [00:02<00:00, 13.20it/s]

Validation DataLoader 0: 100%|██████████| 39/39 [00:02<00:00, 13.26it/s]

                                                                        
Epoch 59: 100%|██████████| 88/88 [00:27<00:00,  3.18it/s, v_num=0]`Trainer.fit` stopped: `max_epochs=60` reached.

Epoch 59: 100%|██████████| 88/88 [00:27<00:00,  3.18it/s, v_num=0]
****************************************************************************************************
Training Finished
Training pid:2746317
*********************************************************